In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [10]:
# Verify wether CUDA is working (though it may just fail at "import torch" if it doesn't)
print("PyTorch version: ", torch.__version__)
print("CUDA available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:        ", torch.cuda.get_device_name(0))
    print("                 ", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

PyTorch version:  2.5.1+cu121
CUDA available:   True
GPU name:         NVIDIA GeForce RTX 5060 Laptop GPU
                  8.546484224 GB


| GPU VRAM | Full Precision (fp32/fp16)     | 8-bit Quantization       | 4-bit Quantization | CPU Offload | Notes                                                       |
| -------- | ------------------------------ | ------------------------ | ------------------ | ----------- | ----------------------------------------------------------- |
| ~4-6 GB  | ❌ Too large                    | ⚠️ Possible but may fail | ✅ Recommended      | ✅ Required  | Only small models (≤3B) fit; use offload for larger models  |
| ~8-10 GB | ⚠️ May fit small fp16 models   | ✅ Recommended            | ✅ Recommended      | ✅ Useful    | Best for mid-size models (3-6B) using 4-bit for larger ones |
| 12-16 GB | ✅ Works for fp16 medium models | ✅ Works                  | ✅ Works            | ⚠️ Optional | Can handle 6-7B models comfortably                          |
| 24-32 GB | ✅ Works for large fp16 models  | ✅ Works                  | ✅ Works            | ⚠️ Optional | Can load 13B–20B models, maybe with 8-bit                   |
| 48+ GB   | ✅ Works for largest models     | ✅ Works                  | ✅ Works            | ⚠️ Optional | Full models (30B+) feasible, often no quantization needed   |


In [13]:
# CPU-only model loading (safe, slow, works for any GPU).
# 4-bit still reduces CPU RAM usage, but no GPU acceleration.

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map={"": "cpu"})


print("Successfully loaded model.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Successfully loaded model.


In [15]:
def generate(prompt):

    if isinstance(prompt, list):
        text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, top_p=0.9, do_sample=True)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [16]:
basic_sql_prompt = """
Act as a BigQuery SQL Expert.
Write a query for the `bigquery-public-data.github_repos.sample_repos` table.
I want to see the repo_name and watch_count for the top 10 most watched repos.
"""

cot_style_prompt = """
Context: I am analyzing the 'bigquery-public-data.github_repos' dataset.
Objective: Identify 'Hot but Fragile' repositories.
Logic:
1. Filter the 'commits' table for repos with > 10,000 total commits.
2. Filter for repos that have fewer than 5 unique contributors in the last 12 months (use committer.time_sec).
3. Join with the 'languages' table to show the primary language.
Output Constraints:
- Use Standard SQL.
- Handle the nested 'committer' record correctly.
- Limit to top 20 by commit count.
"""

chat_style_prompt = [
    {
        "role": "system",
        "content": (
            "You are an expert BigQuery analyst specializing in the bigquery-public-data.github_repos dataset. "
            "Your job is to reason step-by-step "
            "about repository activity, contributors, commit history, and language metadata. "
            "When answering, first explain your reasoning in natural language, then produce clean, correct Standard SQL."
        )
    },
    {
        "role": "user",
        "content": (
            "Identify 'Hot but Fragile' open-source repositories. A repository is 'Hot but Fragile' if:\n"
            "1. It has more than 10,000 total commits.\n"
            "2. It has fewer than 5 unique contributors in the last 12 months.\n"
            "3. It has a primary language listed in the languages table.\n\n"
            "Use the bigquery-public-data.github_repos.commits and bigquery-public-data.github_repos.languages tables. "
            "Handle the nested committer record correctly. "
            "Limit to the top 20 by commit count. "
            "Explain your reasoning before writing SQL."
        )
    }
]


In [ ]:
tests = [
    ("basic_sql_prompt", basic_sql_prompt),
    ("cot_style_prompt", cot_style_prompt),
    ("chat_style_prompt", chat_style_prompt),
]

results = []

print()
for name, prompt in tests:
    output = generate(prompt)
    results.append(output)
    
    print(f"=== {name} ===")
    print(output)
    print()